<a href="https://colab.research.google.com/github/jdasam/ant5015/blob/2025/notebooks/11th_week_automatic_music_transcription.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0. Download Dataset

In [ ]:
!gdown --id 1L99FSzloXGQsmc-OwsVKggvQNF9niN2M
!unzip -q maestro_small.zip

In [ ]:
!pip install mido pretty_midi

## 1. Preprocess Data

### 1.1 Load Dataset Metadata
**Goal**: Load and explore the dataset metadata from JSON file.

**Tasks**:
- Import the `json` library
- Load the JSON file from `'data/data.json'`
- Print the total number of files in the dataset
- Display information about the 50th item to understand the data structure

**Expected output**: You should see metadata containing `audio_filename`, `midi_filename`, and `duration` fields.

In [ ]:
# Your code here

### 1.2 Load Audio and MIDI Files
**Goal**: Load corresponding audio and MIDI files for processing.

**Tasks**:
- Import necessary libraries: `Path`, `IPython.display`, and `torchaudio`
- Create a `data_dir` path object pointing to the 'data/' directory
- Extract audio and MIDI filenames from the metadata (item 50)
- Load the audio file using `torchaudio.load()` (note: change extension to '.flac')
- Store the audio waveform and sample rate

**Expected output**: Audio tensor with shape information and sample rate.

In [ ]:
# Your code here

### 1.3 Verify Audio Properties
**Goal**: Check audio shape and duration consistency.

**Tasks**:
- Print audio shape, sample rate, and expected number of samples
- Play a 20-second clip of the audio using `IPython.display.Audio`

**Expected output**: Audio shape should match duration * sample_rate, and you should hear piano music.

In [ ]:
# Your code here

In [ ]:
# Your code here for audio playback

### 1.4 Load and Explore MIDI File
**Goal**: Load MIDI file and examine its structure.

**Tasks**:
- Import `Message`, `MidiFile`, `MidiTrack` from `mido`
- Load the MIDI file using `MidiFile()`
- Examine the first 20 messages in the merged track

**Expected output**: You should see MIDI messages including note_on, note_off, and control_change events.

In [ ]:
# Your code here

### 1.5 Parse MIDI Notes with Pedal Handling
**Goal**: Convert MIDI messages into note events, properly handling sustain pedal.

**Tasks**:
- Initialize variables for tracking time, notes, and pedal state
- Set pedal threshold to 64 (standard MIDI value)
- Iterate through MIDI messages and:
  - Track cumulative time (convert from ticks to seconds using division by 768)
  - Handle `note_on` events (velocity > 0 for onset, velocity = 0 for offset)
  - Handle `control_change` events for sustain pedal (control = 64)
  - Manage pedal notes that should sustain until pedal release
- Create note dictionaries with 'note', 'velocity', 'onset', and 'offset' fields

**Expected output**: List of note dictionaries with timing information.

In [ ]:
# Your code here - this will be a complex parsing algorithm

### 1.6 Create Piano Roll Representation
**Goal**: Convert parsed notes into piano roll and onset roll matrices.

**Tasks**:
- Import necessary libraries: `ceil` from math, `torch`, `matplotlib.pyplot`
- Set audio processing parameters:
  - Sample rate: 16000 Hz
  - FFT size: 1024
  - Hop size: 512
  - Hop duration: 0.032 seconds (512/16000)
- Calculate total number of time frames based on music duration
- Create two 88×T matrices (piano roll and onset roll) using `torch.zeros`
- Fill matrices by:
  - Converting MIDI note numbers to piano key indices (subtract 21)
  - Converting time to frame indices
  - Setting piano_roll[pitch, onset:offset] = 1 for note duration
  - Setting onset_roll[pitch, onset] = 1 for note onsets
- Combine both rolls: piano_roll += onset_roll
- Visualize the piano roll using `plt.imshow()` for the first 200 frames

**Expected output**: A piano roll visualization showing notes over time.

In [ ]:
# Your code here - piano roll creation

### 1.7 Verify Frame Calculations
**Goal**: Check the frame calculations for debugging.

**Tasks**:
- Print total_num_frame, onset_frame, and offset_frame values

**Expected output**: Frame number values that make sense given the audio duration.

In [ ]:
# Your code here

## 2. Model

### 2.1 Create Spectrogram Converter
**Goal**: Build a neural network module to convert audio to mel-spectrogram.

**Tasks**:
- Import `torch.nn` as `nn`
- Define `SpecConverter` class inheriting from `nn.Module`
- In `__init__`:
  - Create MelSpectrogram transform with:
    - sample_rate=16000
    - n_fft=2048
    - hop_length=512
    - n_mels=352 (88 piano keys × 4 for higher resolution)
  - Create AmplitudeToDB transform
- In `forward` method:
  - Apply mel-spectrogram transform
  - Convert to decibels
  - Normalize by dividing by 80
- Test the converter on loaded audio

**Expected output**: Mel-spectrogram tensor with shape [352, time_frames].

In [ ]:
# Your code here - SpecConverter class

### 2.2 Compare Shapes and Visualize
**Goal**: Verify that spectrogram and piano roll have compatible dimensions.

**Tasks**:
- Print shapes of spectrogram and piano roll
- Print piano roll data type
- Create side-by-side visualization of spectrogram and piano roll
- Use the same time slice (e.g., frames 8000-8200) for comparison

**Expected output**: Both representations should have similar time dimensions and you should see correlation between spectral content and notes.

In [ ]:
# Your code here - shape comparison

In [ ]:
# Your code here - visualization

### 2.3 Create Dataset Class
**Goal**: Build a PyTorch Dataset class for batch training.

**Tasks**:
- Import `random` and `tqdm`
- Define `Dataset` class with methods:
  - `__init__`: Load metadata, set parameters, process all MIDI files, load all audio files
  - `__len__`: Return dataset size
  - `__getitem__`: Return random 15-second slices of audio and corresponding piano roll
  - `load_entire_audio`: Load all audio files using tqdm for progress
  - `process_entire_midi`: Process all MIDI files using tqdm
  - `pre_process_midi`: Process individual MIDI file
  - `make_piano_roll`: Create piano roll from parsed notes
  - `parse_midi_notes`: Parse MIDI messages (same logic as before)
- Set slice_frame to 15 seconds worth of frames
- Use max_data=10 for faster loading during development

**Expected output**: Progress bars showing MIDI and audio loading, and a dataset ready for training.

In [ ]:
# Your code here - Dataset class (this will be long!)

### 2.4 Test Dataset
**Goal**: Verify the dataset works correctly.

**Tasks**:
- Get a sample from the dataset (e.g., index 8)
- Check audio and piano roll shapes
- Play the audio sample
- Visualize the corresponding piano roll

**Expected output**: 15-second audio clip and matching piano roll visualization.

In [ ]:
# Your code here - test dataset

### 2.5 Create DataLoader and Test Batching
**Goal**: Set up PyTorch DataLoader for batch processing.

**Tasks**:
- Create a DataLoader with batch_size=4
- Get one batch using `next(iter(train_loader))`
- Unpack the batch into audios and rolls

**Expected output**: Batched tensors ready for model training.

In [ ]:
# Your code here - DataLoader setup

### 2.6 Define the Transcription Model
**Goal**: Create a CNN-based model for automatic music transcription.

**Tasks**:
- Define `Model` class inheriting from `nn.Module`
- In `__init__`:
  - Initialize SpecConverter
  - Create a sequential CNN with:
    - Conv2d(1, 32, kernel_size=3, padding=1)
    - BatchNorm2d(32) + ReLU + MaxPool2d((2,1))
    - Conv2d(32, 32, kernel_size=3, padding=1)
    - BatchNorm2d(32) + ReLU + MaxPool2d((2,1))
    - Conv2d(32, 32, kernel_size=3, padding=1)
    - BatchNorm2d(32) + ReLU
  - Create linear projection layer: Linear(32 * 88, 88)
- In `forward` method:
  - Convert audio to spectrogram
  - Pass through CNN layers
  - Flatten and permute dimensions
  - Apply linear projection
  - Apply sigmoid activation
- Test model with the batch

**Expected output**: Model output with shape [batch_size, time_frames, 88].

In [ ]:
# Your code here - Model class

### 2.7 Visualize Model Output
**Goal**: Compare model predictions with ground truth.

**Tasks**:
- Create side-by-side plots showing:
  - Model output (predicted piano roll)
  - Ground truth piano roll
- Use the first sample in the batch
- Note: The model is untrained, so predictions will be random

**Expected output**: Two piano roll visualizations for comparison.

In [ ]:
# Your code here - visualization comparison

## 3. Training

### Training Implementation
**Goal**: Implement the training loop for the music transcription model.

**Tasks to implement**:
- Define loss function (Binary Cross Entropy Loss for multi-label classification)
- Set up optimizer (e.g., Adam optimizer)
- Implement training loop with:
  - Forward pass through model
  - Loss calculation
  - Backward pass and parameter updates
  - Loss logging and monitoring
- Add validation loop for model evaluation
- Implement metrics like precision, recall, and F1-score for note detection
- Save model checkpoints during training

**Expected outcome**: A trained model capable of transcribing piano music from audio to MIDI-like representation.

In [ ]:
# Your training code will go here